In [1]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
)
from imagematerials.preprocessing import get_preprocessing_data

In [2]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None)}

In [ ]:
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

time_start = 1971
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1971, 2060, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", climate_scen)
    circular_economy_scenario_dirs = None

    bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs) 


    factory = ModelFactory(
    [bld_sector], complete_timeline
    ).add(GenericStocks, ["buildings"]
    ).add(MaterialIntensities, "buildings",
)
    model = factory.finish()

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    print(f"Finished {scen_id}")

c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev_new\Lib\site-packages\xarray\core\indexing.py:1720: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


IndexError: Time value is not in range of timeline.

In [ ]:
import matplotlib.pyplot as plt


bld_sector.prep_data.get("stocks").sel(Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

bld_sector.prep_data.get("stocks").sel(Type = ['Office',
       'Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')
plt.legend()

In [ ]:
model.buildings.get("inflow").to_array().sel(Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

model.buildings.get("inflow").to_array().sel(Type = ['Office','Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')

plt.legend()

In [ ]:
model.buildings.get("inflow_materials").to_array().sel(material = 'steel', Type = ['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban']).sum(["Region", "Type"]).plot(label = 'residential')

model.buildings.get("inflow_materials").to_array().sel(material = 'steel', Type = ['Office','Retail+', 'Hotels+', 'Govt+']).sum(["Region", "Type"]).plot(label = 'gvt')

plt.legend()

In [ ]:
model.buildings.get("inflow_materials").to_array()

cement_concrete = model.buildings.get("inflow_materials").to_array().sel(material = 'concrete')*0.15


cement_concrete.sum(["Type"]).sel(Region = "CHN").plot()



['Appartment - Rural', 'Appartment - Urban', 'Detached - Rural',
       'Detached - Urban', 'High-rise - Rural', 'High-rise - Urban',
       'Semi-detached - Rural', 'Semi-detached - Urban', 'Office', 'Retail+',
       'Hotels+', 'Govt+']

In [ ]:


# plot concrete use in China
concrete_res = model.buildings.get("inflow_materials").to_array().sel(material = 'concrete').sel(Type = ['Appartment - Rural',
 'Appartment - Urban',
 'Detached - Rural',
 'Detached - Urban',
 'High-rise - Rural',
 'High-rise - Urban',
 'Semi-detached - Rural',
 'Semi-detached - Urban']).sum(["Type"]).loc[2000:].sel(Region = "CHN")
concrete_res = concrete_res*0.10
concrete_res.plot(label = 'cement (concrete) residential')

# plot concrete use in China
concrete_comm = model.buildings.get("inflow_materials").to_array().sel(material = 'concrete').sel(Type = ['Office', 'Retail+',
       'Hotels+', 'Govt+']).sum(["Type"]).sel(Region = "CHN").loc[2000:]
concrete_comm = concrete_comm*0.10
concrete_comm.plot(label = 'cement (concrete) commercial')



plt.legend()

In [ ]:


# plot concrete use in China
concrete_res = model.buildings.get("inflow_materials").to_array().sel(material = 'concrete').sel(Type = ['Appartment - Rural',
 'Appartment - Urban',
 'Detached - Rural',
 'Detached - Urban',
 'High-rise - Rural',
 'High-rise - Urban',
 'Semi-detached - Rural',
 'Semi-detached - Urban']).sum(["Type"]).loc[2000:].sel(Region = "CHN")
concrete_res = concrete_res*0.12
concrete_res = concrete_res.pint.to('megatonne')
concrete_res.plot(label = 'cement (concrete) residential')

# plot concrete use in China
concrete_comm = model.buildings.get("inflow_materials").to_array().sel(material = 'concrete').sel(Type = [
 'Office', 'Retail+',
       'Hotels+', 'Govt+']).sum(["Type"]).sel(Region = "CHN").loc[2000:]
concrete_comm = concrete_comm*0.12
concrete_comm = concrete_comm.pint.to('megatonne')
concrete_comm.plot(label = 'cement (concrete) commercial')

total = concrete_res + concrete_comm
total = total.pint.to('megatonne')
total.plot(label = 'cement (concrete) total')

plt.legend()

In [ ]:
inflow_m2 = model.buildings.get("inflow").to_array().sel(Region = 'CHN').sum("Type").loc[2020:]
inflow_m2 = inflow_m2.pint.to("km**2")
inflow_m2.plot()

In [ ]:
stock = model.buildings.get("stocks").sel(Region = 'CHN').sum("Type").loc[2020:]
stock = stock.pint.to("km**2")

stock.plot()


In [ ]:
model.buildings.keys()